In [1]:
MINIKUBE_IP = '192.168.58.2'

In [2]:
# to define ip run: kubectl get svc -n minio-dev
STORAGE_URI = "http://10.107.38.26:6544"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"
CATALOG_URI = "http://10.109.108.198:6788/api/v1"
WAREHOUSE = "s3a://iceberg/"

In [3]:
from pyspark.sql import SparkSession

In [4]:
# .config("spark.jars.packages","/home/greg/drivers/jars/postgresql-42.7.3.jar") \

# .config('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO') \

spark = SparkSession.builder \
    .appName("SparkK8sTest") \
    .master(f"k8s://https://{MINIKUBE_IP}:8443") \
    .config("spark.kubernetes.container.image", "spark-custom:3.5.8") \
    .config("spark.kubernetes.namespace", "spark") \
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark") \
    .config("spark.kubernetes.authenticate.executor.serviceAccountName", "spark") \
    .config("spark.jars","/home/greg/drivers/jars/postgresql-42.7.3.jar,"
                         "/home/greg/drivers/jars/hadoop-aws-3.3.4.jar,"
                         "/home/greg/drivers/jars/aws-java-sdk-bundle-1.12.262.jar,"
                         "/home/greg/drivers/jars/iceberg-aws-bundle-1.9.2.jar,"
                         "/home/greg/drivers/jars/iceberg-spark-runtime-3.5_2.12-1.9.2.jar,"
                         "/home/greg/drivers/jars/nessie-spark-extensions-3.5_2.12-0.105.3.jar") \
    .config('spark.sql.extensions', 
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
            'org.projectnessie.spark.extensions.NessieSparkSessionExtensions') \
    .config('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.nessie.uri', CATALOG_URI) \
    .config('spark.sql.catalog.nessie.ref', 'main') \
    .config('spark.sql.catalog.nessie.authentication.type', 'NONE') \
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog') \
    .config('spark.sql.catalog.nessie.s3.endpoint', STORAGE_URI) \
    .config('spark.sql.catalog.nessie.warehouse', WAREHOUSE) \
    .config("spark.hadoop.fs.s3a.endpoint", STORAGE_URI) \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print("Spark Running")

26/02/23 20:04:16 WARN Utils: Your hostname, ls02 resolves to a loopback address: 127.0.1.1; using 192.168.1.6 instead (on interface wlp5s0)
26/02/23 20:04:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/02/23 20:04:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark Running


26/02/23 20:04:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
# df = spark.range(1, 1000000)
# df.count()

In [5]:
# jdbc_url = "jdbc:postgresql://postgres.psql-dev.svc.cluster.local:5432/postgres"
jdbc_url = f"jdbc:postgresql://{MINIKUBE_IP}:32432/postgres"
properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

In [6]:
sales_df = spark.read.jdbc(url=jdbc_url, table="fashion_sales", properties=properties)

In [7]:
sales_df.show()

+---+---------------+---------+------------+----------+--------------+------------------+-------------+
| id|   product_name| category|sales_amount|sales_date|store_location|customer_age_group|campaign_name|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+
|  1| Slim Fit Jeans|    Denim|       89.99|2024-03-01|      New York|             18-24|Spring Launch|
|  2| Leather Jacket|Outerwear|      249.99|2024-03-01|   Los Angeles|             25-34|Spring Launch|
|  3|Graphic T-Shirt|     Tops|       39.99|2024-03-02|       Chicago|             18-24|March Madness|
|  4|   Summer Dress|  Dresses|      129.99|2024-03-03|      New York|             35-44|March Madness|
|  5|Casual Sneakers| Footwear|       99.99|2024-03-03|   Los Angeles|             25-34|Spring Launch|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+



In [8]:
BUCKET_NAME = "src"
FILE_PATH = "data/parquet/part-00000.snappy.parquet"

s3a_path = f"s3a://{BUCKET_NAME}/{FILE_PATH}"

df = spark.read.parquet(s3a_path)

df.show()

26/02/23 20:05:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

+---+---------------+---------+------------+----------+--------------+------------------+-------------+
| id|   product_name| category|sales_amount|sales_date|store_location|customer_age_group|campaign_name|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+
|  1| Slim Fit Jeans|    Denim|       89.99|2024-03-01|      New York|             18-24|Spring Launch|
|  2| Leather Jacket|Outerwear|      249.99|2024-03-01|   Los Angeles|             25-34|Spring Launch|
|  3|Graphic T-Shirt|     Tops|       39.99|2024-03-02|       Chicago|             18-24|March Madness|
|  4|   Summer Dress|  Dresses|      129.99|2024-03-03|      New York|             35-44|March Madness|
|  5|Casual Sneakers| Footwear|       99.99|2024-03-03|   Los Angeles|             25-34|Spring Launch|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+



In [9]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.sales;")

DataFrame[]

In [10]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.special;")

DataFrame[]

In [12]:
spark.read.table("nessie.sales.fashion_sales").show()

[Stage 3:>                                                          (0 + 1) / 1]

+---+---------------+---------+------------+----------+--------------+------------------+-------------+
| id|   product_name| category|sales_amount|sales_date|store_location|customer_age_group|campaign_name|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+
|  1| Slim Fit Jeans|    Denim|       89.99|2024-03-01|      New York|             18-24|Spring Launch|
|  2| Leather Jacket|Outerwear|      249.99|2024-03-01|   Los Angeles|             25-34|Spring Launch|
|  3|Graphic T-Shirt|     Tops|       39.99|2024-03-02|       Chicago|             18-24|March Madness|
|  4|   Summer Dress|  Dresses|      129.99|2024-03-03|      New York|             35-44|March Madness|
|  5|Casual Sneakers| Footwear|       99.99|2024-03-03|   Los Angeles|             25-34|Spring Launch|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+



In [13]:
sales_df.writeTo("nessie.special.fashion_sales_new").createOrReplace()

In [14]:
spark.read.table("nessie.special.fashion_sales_new").show()

+---+---------------+---------+------------+----------+--------------+------------------+-------------+
| id|   product_name| category|sales_amount|sales_date|store_location|customer_age_group|campaign_name|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+
|  1| Slim Fit Jeans|    Denim|       89.99|2024-03-01|      New York|             18-24|Spring Launch|
|  2| Leather Jacket|Outerwear|      249.99|2024-03-01|   Los Angeles|             25-34|Spring Launch|
|  3|Graphic T-Shirt|     Tops|       39.99|2024-03-02|       Chicago|             18-24|March Madness|
|  4|   Summer Dress|  Dresses|      129.99|2024-03-03|      New York|             35-44|March Madness|
|  5|Casual Sneakers| Footwear|       99.99|2024-03-03|   Los Angeles|             25-34|Spring Launch|
+---+---------------+---------+------------+----------+--------------+------------------+-------------+



In [16]:
spark.stop()